#환경 세팅

In [ ]:
!pip install "paramiko<3.0" sshtunnel --upgrade-strategy eager --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.1/213.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.6 MB/s eta 0:00:00


In [ ]:
# !pip install sshtunnel psycopg2-binary

In [ ]:
# 1. 필요한 라이브러리를 임포트합니다.
from sshtunnel import SSHTunnelForwarder
import psycopg2
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import bigquery
from google.cloud import storage
import gspread
from google.auth import default
from google.auth.transport.requests import Request
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
import os
import gcsfs
import io
import paramiko
import json

/usr/local/lib/python3.12/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/local/lib/python3.12/dist-packages/paramiko/transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


#인증 정보

In [ ]:

# ... (기존 GCS 및 DB 접속 정보 설정은 동일) ...
ssh_host = '15.164.127.160'
ssh_username = 'ec2-user'
gcs_pem_path = 'gs://fs-aws-pem/aws-eb.pem'

# 1. GCS에서 PEM 파일 내용을 문자열로 바로 읽어옵니다.
fs = gcsfs.GCSFileSystem()
with fs.open(gcs_pem_path, 'rb') as f:
    pem_file_content = f.read().decode('utf-8') # 바이트를 문자열로 디코딩

# 2. 문자열 내용을 paramiko가 인식할 수 있는 PKey 객체로 변환합니다.
#    io.StringIO를 사용해 문자열을 파일처럼 다룹니다.
pkey = paramiko.RSAKey.from_private_key(io.StringIO(pem_file_content))


# ... (DB 접속 정보 설정은 동일) ...
db_host = 'fvsp-prd-rds-main-01.cluster-ro-c5i6uppx3nsn.ap-northeast-2.rds.amazonaws.com'
db_port = 5432
db_user = 'readonly_user_mkr'
db_password = '_897iKSCenA1'
db_name = 'fivespot'

# 기존 구매자

In [ ]:


# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체를 직접 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # ... (이후 DB 연결 및 쿼리 실행 코드는 동일) ...
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_oldUser = pd.read_sql("""
                            SELECT
                                uid,
                                phone,
                                user_login,
                                user_name,
                                TO_CHAR(regdate AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS signup_date
                            FROM fivespot_user;
                    """, conn)
    conn.close()
df_oldUser.head()


  df_oldUser = pd.read_sql("""



,uid,phone,user_login,user_name,signup_date
0,19,010-9726-8826,gt.kim@fastfive.co.kr,김규태패스트파이브,2021-11-21
1,27,010-7400-8232,harry.park@fastfive.co.kr,해리,2021-11-21
2,37,010-8430-6295,ff-pass@fastfive.co.kr,패파어드민,2021-10-26
3,40,010-2792-3875,ethan.dy.kim@gmail.com,김동영,None
4,41,010-9367-0577,panda0705@gmail.com,주지연,None


In [ ]:


# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체를 직접 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # ... (이후 DB 연결 및 쿼리 실행 코드는 동일) ...
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_oldPayment = pd.read_sql("""
      SELECT
            TO_CHAR(p.regdate AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS p_date,
            p.order_id,
            o.uid,
            (p.content::json) ->> 'name' AS name,
            p.total,
            c.contract_type,
            c.start_date,
            c.end_date
        FROM
            fivespot_payment p
        LEFT JOIN
            fivespot_order o ON p.order_id = o.order_id
        LEFT JOIN
            fivespot_contract c ON o.contract_id = c.contract_id
        WHERE
            p.status = 'buy'
            AND p.total > 0
            AND c.contract_type != 'service'
        ORDER BY
            p_date DESC;
                    """, conn)
    conn.close()
df_oldPayment.head()


  df_oldPayment = pd.read_sql("""



,p_date,order_id,uid,name,total,contract_type,start_date,end_date
0,2025-06-10,139317,25571,차감형 패스 1회,30000,voucher,2025-06-10,2025-06-30
1,2025-06-10,139325,26420,차감형 패스 1회,30000,voucher,2025-06-10,2025-06-30
2,2025-06-10,139331,43466,차감형 패스 1회,30000,voucher,2025-06-10,2025-06-30
3,2025-06-10,139332,27726,차감형 패스 1회,30000,voucher,2025-06-10,2025-06-30
4,2025-06-10,139333,43467,차감형 패스 1회,30000,voucher,2025-06-10,2025-06-30


In [ ]:
df_oldPayment_sum = df_oldPayment.groupby('uid')['total'].sum().reset_index()
df_oldUser = df_oldUser.merge(df_oldPayment_sum, on='uid', how='left')
df_oldPurchaser = df_oldUser[df_oldUser['total'].notna()]
df_oldPurchaser['phone_number'] = df_oldPurchaser['phone'].str.replace('-', '', regex=False)
df_oldPurchaser_phone = df_oldPurchaser.groupby('phone_number')['total'].sum().reset_index()
df_oldPurchaser_phone['기존구매자'] = "기존구매자"
df_oldPurchaser_phone.head()

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_oldPurchaser['phone_number'] = df_oldPurchaser['phone'].str.replace('-', '', regex=False)



,phone_number,total,기존구매자
0,,1017000.0,기존구매자
1,(206) 557051,30000.0,기존구매자
2,(240) 579708,250000.0,기존구매자
3,(240) 801213,30000.0,기존구매자
4,(404) 909701,279000.0,기존구매자


# User Raw

In [ ]:


# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체를 직접 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # ... (이후 DB 연결 및 쿼리 실행 코드는 동일) ...
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_user = pd.read_sql("""
                             SELECT
                                TO_CHAR(u.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS signup_date,
                                u.client_uid AS uid,
                                u.name,
                                u.phone_number,
                                ui.email,
                                ui.status,
                                ua.is_agreed
                            FROM client u
                            LEFT JOIN client_info ui
                                ON u.client_uid = ui.client_uid
                            LEFT JOIN (
                                SELECT ca.*
                                FROM client_agreement ca
                                INNER JOIN (
                                    SELECT client_uid, MAX(agreement_uid) AS max_agreement_uid
                                    FROM client_agreement
                                    GROUP BY client_uid
                                ) latest
                                ON ca.client_uid = latest.client_uid AND ca.agreement_uid = latest.max_agreement_uid
                            ) ua
                                ON u.client_uid = ua.client_uid
                            WHERE ui.status != 'DELETED';
                    """, conn)
    conn.close()
df_user.head()


  df_user = pd.read_sql("""



,signup_date,uid,name,phone_number,email,status,is_agreed
0,2025-06-11,61,김혜민,01063205446,kimhm2029@gmail.com,ACTIVE,True
1,2025-06-11,62,민선영,01073178240,sthstars@naver.com,ACTIVE,True
2,2025-06-11,63,수빈빈,01071971584,k_subin@naver.com,ACTIVE,True
3,2025-06-15,1247,승균,01064266492,jinggun98@gmail.com,ACTIVE,True
4,2025-06-11,66,최강우,01026185886,kangwooc@gmail.com,ACTIVE,True


In [ ]:


# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체를 직접 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # ... (이후 DB 연결 및 쿼리 실행 코드는 동일) ...
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_admin = pd.read_sql("""
                             SELECT
                                admin_uid,
                                phone_number,
                                email,
                                name
                            FROM admin;
                    """, conn)
    conn.close()
df_admin['admin'] = 'admin'
df_admin.head()

  df_admin = pd.read_sql("""



,admin_uid,phone_number,email,name,admin
0,2,01057808356,bm.gu@fastfive.co.kr,구보미,admin
1,3,01093234291,bh.yoo@fastfive.co.kr,유병호,admin
2,5,01071971584,soobin.k@fastfive.co.kr,김수빈,admin
3,8,01099151119,jh.heo@fastfive.co.kr,허지혜,admin
4,7,01052613762,sb.kim@fastfive.co.kr,김상백,admin


In [ ]:
df_user = df_user.merge(df_admin[['phone_number', 'admin']], on='phone_number', how='left')
df_user = df_user.merge(df_oldPurchaser_phone[['phone_number', '기존구매자']], on='phone_number', how='left')
df_user = df_user[df_user['admin'].isna()]
df_user.head()

,signup_date,uid,name,phone_number,email,status,is_agreed,admin,기존구매자
0,2025-06-11,61,김혜민,01063205446,kimhm2029@gmail.com,ACTIVE,True,NaN,기존구매자
1,2025-06-11,62,민선영,01073178240,sthstars@naver.com,ACTIVE,True,NaN,기존구매자
3,2025-06-15,1247,승균,01064266492,jinggun98@gmail.com,ACTIVE,True,NaN,기존구매자
8,2025-06-15,1280,조승규,01090522904,skps2000@gmail.com,ACTIVE,True,NaN,기존구매자
9,2025-07-18,4508,이성호,01044991052,soung2005@naver.com,ACTIVE,True,NaN,기존구매자


# 결제 Raw

In [ ]:


# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체를 직접 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # ... (이후 DB 연결 및 쿼리 실행 코드는 동일) ...
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_payment = pd.read_sql("""
                        SELECT

                        TO_CHAR(c.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS c_date,
                        c.contract_uid transaction_id,
                        c.client_uid uid,
                        c.status,
                        c.product_name,
                        pp.name product_period,
                        TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                        TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                        TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
                        ph.price,
                        TO_CHAR(c.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS created_at,
                        c.is_migrated,
                        p.order_id
                        FROM contract c
                        LEFT JOIN payment p
                        ON c.contract_uid = p.contract_payment_uid
                        LEFT JOIN price_policy pp
                        ON c.price_policy_uid = pp.price_policy_uid
                        LEFT JOIN payment_history ph
                        ON c.contract_uid = ph.payment_uid
                        WHERE ph.payment_status = 'DONE'
                        ORDER BY created_at ASC;
                    """, conn)
    conn.close()
df_payment.head()

  df_payment = pd.read_sql("""



,c_date,transaction_id,uid,status,product_name,product_period,start_date,initial_end_date,actual_end_date,price,created_at,is_migrated,order_id
0,2025-06-11,113,66,EXPIRED,무제한 패스,1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,True,78bb7e25-ca6b-43cc-9829-beb8dbbd38a7
1,2025-06-11,114,63,EXPIRED,차감형 패스,10회,2025-06-10,2025-09-30,2025-09-30,0.0,2025-06-11 00:57:28,True,0a5da319-441b-48a7-b3e7-cb62634bd103
2,2025-06-11,115,64,EXPIRED,주말&야간 패스,1개월,2025-06-10,2025-07-10,2025-07-11,0.0,2025-06-11 00:57:39,True,e00129b2-4960-4a6f-87a6-969f8a3aae1d
3,2025-06-11,116,65,ACTIVE,무제한 패스,12개월,2024-10-19,2025-11-15,2025-11-15,0.0,2025-06-11 00:57:44,True,14f62023-da73-4711-b970-8ce592555c8b
4,2025-06-11,117,62,EXPIRED,주말&야간 패스,1개월,2025-06-01,2025-07-01,2025-07-02,169000.0,2025-06-11 00:58:36,True,5d771bad-5186-4b9e-9536-5756450ef6aa


In [ ]:
df_payment['product_name'] = df_payment['product_name'] + ' - ' + df_payment['product_period'].astype(str)
df_payment = df_payment.drop(columns=['product_period'])
df_payment.head()

,c_date,transaction_id,uid,status,product_name,start_date,initial_end_date,actual_end_date,price,created_at,is_migrated,order_id
0,2025-06-11,113,66,EXPIRED,무제한 패스 - 1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,True,78bb7e25-ca6b-43cc-9829-beb8dbbd38a7
1,2025-06-11,114,63,EXPIRED,차감형 패스 - 10회,2025-06-10,2025-09-30,2025-09-30,0.0,2025-06-11 00:57:28,True,0a5da319-441b-48a7-b3e7-cb62634bd103
2,2025-06-11,115,64,EXPIRED,주말&야간 패스 - 1개월,2025-06-10,2025-07-10,2025-07-11,0.0,2025-06-11 00:57:39,True,e00129b2-4960-4a6f-87a6-969f8a3aae1d
3,2025-06-11,116,65,ACTIVE,무제한 패스 - 12개월,2024-10-19,2025-11-15,2025-11-15,0.0,2025-06-11 00:57:44,True,14f62023-da73-4711-b970-8ce592555c8b
4,2025-06-11,117,62,EXPIRED,주말&야간 패스 - 1개월,2025-06-01,2025-07-01,2025-07-02,169000.0,2025-06-11 00:58:36,True,5d771bad-5186-4b9e-9536-5756450ef6aa


In [ ]:
df_payment['row_num'] = df_payment.groupby('uid').cumcount() + 1
df_user_dedup = df_user.drop_duplicates(subset='uid', keep='first')
df_payment = df_payment.merge(df_user_dedup[['uid', '기존구매자']], on='uid', how='left')

# 구매 유형 분류 함수 정의
def classify_purchase(row):
    if (row['기존구매자'] == '기존구매자') or (row['row_num'] > 1):
        return '재구매'
    else:
        return '첫구매'

# 적용
df_payment['purchase_type'] = df_payment.apply(classify_purchase, axis=1)


In [ ]:
df_payment.head()

,c_date,transaction_id,uid,status,product_name,start_date,initial_end_date,actual_end_date,price,created_at,is_migrated,order_id,row_num,기존구매자,purchase_type
0,2025-06-11,113,66,EXPIRED,무제한 패스 - 1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,True,78bb7e25-ca6b-43cc-9829-beb8dbbd38a7,1,NaN,첫구매
1,2025-06-11,114,63,EXPIRED,차감형 패스 - 10회,2025-06-10,2025-09-30,2025-09-30,0.0,2025-06-11 00:57:28,True,0a5da319-441b-48a7-b3e7-cb62634bd103,1,NaN,첫구매
2,2025-06-11,115,64,EXPIRED,주말&야간 패스 - 1개월,2025-06-10,2025-07-10,2025-07-11,0.0,2025-06-11 00:57:39,True,e00129b2-4960-4a6f-87a6-969f8a3aae1d,1,NaN,첫구매
3,2025-06-11,116,65,ACTIVE,무제한 패스 - 12개월,2024-10-19,2025-11-15,2025-11-15,0.0,2025-06-11 00:57:44,True,14f62023-da73-4711-b970-8ce592555c8b,1,NaN,첫구매
4,2025-06-11,117,62,EXPIRED,주말&야간 패스 - 1개월,2025-06-01,2025-07-01,2025-07-02,169000.0,2025-06-11 00:58:36,True,5d771bad-5186-4b9e-9536-5756450ef6aa,1,기존구매자,재구매


#빅쿼리 데이터와 DB 데이터 병합

In [ ]:
client = bigquery.Client()

sql_query = """
    SELECT *
    FROM `fivespot-bigquery.MKT.MKT_eventAll_web` AS t1
    LEFT JOIN `fivespot-bigquery.MKT.MKT_userId_match_web` AS t2
    ON t1.user_pseudo_id = t2.user_pseudo_id
"""
df = client.query(sql_query).to_dataframe()
df = df.dropna(subset=['user_id'])


df['row_num2'] = df.sort_values(by=['user_id', 'event_timestamp']).groupby('user_id').cumcount() + 1
df = df.sort_values(by=['user_id', 'row_num2'], ascending=[True, True])



df_payment['uid'] = df_payment['uid'].astype(str)
df_payment = df_payment[df_payment['c_date'] >= '2025-06-11']
df_purchase = pd.merge(df_payment, df, left_on='uid', right_on='user_id', how='left')

df_purchase.loc[
    df_purchase['event_content'] == 'fs2147-250728-ua-megasale-7-lyj',
    ['event_source', 'event_medium']
] = ['mms', 'paid']

df_purchase.head()



,c_date,transaction_id,uid,status,product_name,start_date,initial_end_date,actual_end_date,price,created_at,...,event_medium,event_campaign,event_content,event_term,row_num_y,event_timestamp,traffic_type,user_pseudo_id_1,user_id,row_num2
0,2025-06-11,113,66,EXPIRED,무제한 패스 - 1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,...,referral,(referral),None,None,1,1755445148137457,None,jBofpssg/ba/RZcmcyO4WTlV6IIkyLw2wpzcHpBRIV8=.1...,66,1.0
1,2025-06-11,113,66,EXPIRED,무제한 패스 - 1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,...,referral,(referral),None,None,2,1755445148137457,None,jBofpssg/ba/RZcmcyO4WTlV6IIkyLw2wpzcHpBRIV8=.1...,66,2.0
2,2025-06-11,113,66,EXPIRED,무제한 패스 - 1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,...,referral,(referral),None,None,4,1755494533155985,None,jBofpssg/ba/RZcmcyO4WTlV6IIkyLw2wpzcHpBRIV8=.1...,66,3.0
3,2025-06-11,113,66,EXPIRED,무제한 패스 - 1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,...,referral,(referral),None,None,3,1755494533155985,None,jBofpssg/ba/RZcmcyO4WTlV6IIkyLw2wpzcHpBRIV8=.1...,66,4.0
4,2025-06-11,113,66,EXPIRED,무제한 패스 - 1개월,2025-06-10,2025-07-10,2025-07-10,0.0,2025-06-11 00:54:47,...,referral,(referral),None,None,5,1756083110664795,None,jBofpssg/ba/RZcmcyO4WTlV6IIkyLw2wpzcHpBRIV8=.1...,66,5.0


In [ ]:
df_purchase['event_source'] = df_purchase['event_source'].astype(str)
# 'Media' 컬럼 생성 규칙 적용
def categorize_media(row):
    # 'event_source'가 None이 아닌 경우에만 체크
    if row['event_source'] in ['fbig', 'fb', 'ig']:
        return 'FBIG'
    elif row['event_source'] in ['tiktok']:
        return 'Tiktok'
    elif row['event_source'] in ['kakao']:
        return 'Kakao'
    elif row['event_source'] in ['toss']:
        return 'Toss'
    elif row['event_source'] in ['navergfa', 'naver_gfa']:
        return 'NaverGFA'
    elif row['event_source'] and ('blog' in row['event_source'] or 'brunch' in row['event_source'] or 'tistory' in row['event_source']):
        return 'Blog'
    elif row['event_source'] in ['mms','alimtalk']:
        return 'CRM'
    elif row['event_source'] and 'daum' in row['event_source'] and row['event_campaign'] == 'sa':
        return 'DaumSA'
    elif row['event_source'] and 'naver' in row['event_source'] and row['event_campaign'] in ['searchads','sa']:
        return 'NaverSA'
    elif row['event_source'] and 'naver' in row['event_source'] and row['event_campaign'] in ['brandsearch','ba']:
        return 'NaverBSA'
    elif row['event_source'] and 'daum' in row['event_source'] and row['event_medium'] == 'organic':
        return 'DaumSEO'
    elif row['event_source'] == 'google' and row['event_medium'] == 'organic':
        return 'GoogleSEO'
    elif row['event_source'] == 'google' and row['event_campaign'] in ['searchads', '19420764713', '17302820061']:
        return 'GoogleSA'
    # 이전 조건에 해당하지 않고 'event_source'가 정확히 'google'인 경우
    elif row['event_source'] == 'google':
        return 'GoogleDA'
    # 'event_source'에 'place'가 포함되고 'kakao'가 포함되지 않은 경우
    elif row['event_source'] and 'place' in row['event_source'] and 'kakao' not in row['event_source']:
        return 'NaverPlace'
    elif row['event_source'] and 'naver' in row['event_source']:
        return 'NaverSEO'
    elif row['event_source'] in ['fastfive.co.kr']:
        return 'fastfive.co.kr'
    elif row['event_source'] in ['app']:
        return 'ffapp'
    elif row['event_source'] in ['workanywhere.co.kr','workanywhere']:
        return 'workanywhere.co.kr'
    elif row['event_source'] in ['oopy','fivespot.oopy.io']:
        return 'oopy'
    else:
        return 'ETC'

# 'Media' 컬럼 생성
df_purchase['Media'] = df_purchase.apply(categorize_media, axis=1)


In [ ]:
# 1. 구매시간(created_at) → datetime으로 변환 후 KST timezone 지정
df_purchase['created_at_dt'] = pd.to_datetime(df_purchase['created_at'])
df_purchase['created_at_dt'] = df_purchase['created_at_dt'].dt.tz_localize('Asia/Seoul')

# (수정) 2. pd.to_datetime 호출 전, 유효하지 않은 타임스탬프 값을 미리 제거
# 타임스탬프는 일반적으로 양수이므로 음수 값을 가진 행을 먼저 제거합니다.
df_purchase = df_purchase[df_purchase['event_timestamp'] >= 0]

# 이제 안전하게 datetime으로 변환합니다.
df_purchase['event_time_dt'] = pd.to_datetime(df_purchase['event_timestamp'], unit='us', errors='coerce')

# NaT 값 행 제거 (혹시 모를 다른 변환 오류 값 처리)
df_purchase.dropna(subset=['event_time_dt'], inplace=True)

# 시간대 변환
df_purchase['event_time_dt'] = df_purchase['event_time_dt'].dt.tz_localize('UTC')
df_purchase['event_time_kst'] = df_purchase['event_time_dt'].dt.tz_convert('Asia/Seoul')

# 3. 방문 시간(event_time_kst)이 구매 시간(created_at_dt)보다 늦은 경우 제거
df_purchase = df_purchase[df_purchase['event_time_kst'] <= df_purchase['created_at_dt']]

In [ ]:
# 제거할 도메인 리스트
exclude_sources = [
    'ffspot.co.kr',
    'logins.daum.net',
    'kauth.kakao.com',
    'fivespot.channel.io',
    'accounts.kakao.com',
    'fivespot.oopy.io',
    'p745j.channel.io',
    'ksmobile.inicis.com',
    'form.jotform.com',
    'payment-gateway.tosspayments.com',
    'google-play',
    'payment-widget.tosspayments.com'
]

# event_source가 리스트에 없는 행만 남김
df_purchase = df_purchase[~df_purchase['event_source'].isin(exclude_sources)]
df_purchase = df_purchase[~df_purchase['is_migrated']]

df_purchase.head()

,c_date,transaction_id,uid,status,product_name,start_date,initial_end_date,actual_end_date,price,created_at,...,row_num_y,event_timestamp,traffic_type,user_pseudo_id_1,user_id,row_num2,Media,created_at_dt,event_time_dt,event_time_kst
3940,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,1,1749596034672101,None,161602292.1749596035,85,1.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:54.672101+00:00,2025-06-11 07:53:54.672101+09:00
3941,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,2,1749596034672101,None,161602292.1749596035,85,2.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:54.672101+00:00,2025-06-11 07:53:54.672101+09:00
3942,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,3,1749596034672101,None,161602292.1749596035,85,3.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:54.672101+00:00,2025-06-11 07:53:54.672101+09:00
3943,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,4,1749596036924766,None,161602292.1749596035,85,4.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:56.924766+00:00,2025-06-11 07:53:56.924766+09:00
7520,2025-06-11,255,122,CANCELED,2시간 패스 - 1회,None,None,None,5000.0,2025-06-11 09:02:23,...,1,1749597740144725,None,944027773.1743328104,122,1.0,ffapp,2025-06-11 09:02:23+09:00,2025-06-10 23:22:20.144725+00:00,2025-06-11 08:22:20.144725+09:00


### Lookback window

In [ ]:
# ConvDuration 컬럼 생성 (일 단위 차이)
df_purchase['ConvDuration'] = (df_purchase['created_at_dt'] - df_purchase['event_time_kst']).dt.days
df_purchase = df_purchase[(df_purchase['ConvDuration'] >= 0) & (df_purchase['ConvDuration'] < 8)]


# 다채널 Data Driven

In [ ]:


# 1. row_num2 숫자 변환
df_purchase['row_num2'] = pd.to_numeric(df_purchase['row_num2'], errors='coerce')

# 2. uid별 최대 row_num2 구하기
max_row_num2 = df_purchase.groupby('uid')['row_num2'].transform('max')

# 3. 최대값 행만 필터링
df_LastClick = df_purchase[df_purchase['row_num2'] == max_row_num2].copy()
df_LastClick.head()

,c_date,transaction_id,uid,status,product_name,start_date,initial_end_date,actual_end_date,price,created_at,...,event_timestamp,traffic_type,user_pseudo_id_1,user_id,row_num2,Media,created_at_dt,event_time_dt,event_time_kst,ConvDuration
8411,2025-06-11,272,161,ACTIVE,무제한 패스 - 12개월,2025-06-11,2026-06-10,2026-06-10,1684800.0,2025-06-11 09:23:56,...,1749600575627112,None,446955329.1749521987,161,3.0,NaverSEO,2025-06-11 09:23:56+09:00,2025-06-11 00:09:35.627112+00:00,2025-06-11 09:09:35.627112+09:00,0
11582,2025-06-11,308,199,CANCELED,12시간 패스 - 1회,2025-06-13,2025-06-13,2025-06-13,20000.0,2025-06-11 10:14:56,...,1749603439540369,None,29665800.1749601879,199,4.0,ETC,2025-06-11 10:14:56+09:00,2025-06-11 00:57:19.540369+00:00,2025-06-11 09:57:19.540369+09:00,0
15473,2025-06-11,392,262,EXPIRED,6시간 패스 - 1회,2025-06-11,2025-06-11,2025-06-11,12000.0,2025-06-11 11:48:16,...,1749609967168885,None,22496121.1728353030,262,6.0,NaverBSA,2025-06-11 11:48:16+09:00,2025-06-11 02:46:07.168885+00:00,2025-06-11 11:46:07.168885+09:00,0
16673,2025-06-11,432,293,EXPIRED,8시간 패스 - 1회,2025-06-12,2025-06-12,2025-06-12,15000.0,2025-06-11 12:09:40,...,1749610476484732,None,726966942.1749610476,293,3.0,fastfive.co.kr,2025-06-11 12:09:40+09:00,2025-06-11 02:54:36.484732+00:00,2025-06-11 11:54:36.484732+09:00,0
17037,2025-06-11,445,297,EXPIRED,주말&야간 패스 - 1개월,2025-06-11,2025-07-10,2025-07-11,0.0,2025-06-11 12:31:14,...,1749611460607145,None,2031678672.1749611460,297,3.0,NaverSEO,2025-06-11 12:31:14+09:00,2025-06-11 03:11:00.607145+00:00,2025-06-11 12:11:00.607145+09:00,0


In [ ]:
df_LastClick = df_LastClick.drop(columns=[
    'created_at',
    'is_migrated',
    'row_num_x',
    'row_num_y',
    'event_timestamp',
    'traffic_type',
    'user_pseudo_id_1',
    'user_id',
    'event_time_dt'
])


In [ ]:


# 2. uid별 최소 row_num2 구하기
min_row_num2 = df_purchase.groupby('uid')['row_num2'].transform('min')

# 3. 최소값 행만 필터링
df_1stClick = df_purchase[df_purchase['row_num2'] == min_row_num2].copy()

df_1stClick.head()


,c_date,transaction_id,uid,status,product_name,start_date,initial_end_date,actual_end_date,price,created_at,...,event_timestamp,traffic_type,user_pseudo_id_1,user_id,row_num2,Media,created_at_dt,event_time_dt,event_time_kst,ConvDuration
3940,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,1749596034672101,None,161602292.1749596035,85,1.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:54.672101+00:00,2025-06-11 07:53:54.672101+09:00,0
7520,2025-06-11,255,122,CANCELED,2시간 패스 - 1회,None,None,None,5000.0,2025-06-11 09:02:23,...,1749597740144725,None,944027773.1743328104,122,1.0,ffapp,2025-06-11 09:02:23+09:00,2025-06-10 23:22:20.144725+00:00,2025-06-11 08:22:20.144725+09:00,0
7807,2025-06-11,262,153,EXPIRED,8시간 패스 - 1회,2025-06-11,2025-06-11,2025-06-11,15000.0,2025-06-11 09:14:57,...,1749600008062536,None,110999051.1749600007,153,1.0,NaverSEO,2025-06-11 09:14:57+09:00,2025-06-11 00:00:08.062536+00:00,2025-06-11 09:00:08.062536+09:00,0
8409,2025-06-11,272,161,ACTIVE,무제한 패스 - 12개월,2025-06-11,2026-06-10,2026-06-10,1684800.0,2025-06-11 09:23:56,...,1749600575627112,None,446955329.1749521987,161,1.0,NaverSEO,2025-06-11 09:23:56+09:00,2025-06-11 00:09:35.627112+00:00,2025-06-11 09:09:35.627112+09:00,0
9104,2025-06-11,295,175,TERMINATION,무제한 패스 - 12개월,2025-07-04,2026-07-03,2025-11-26,0.0,2025-06-11 09:50:48,...,1749600625243602,None,1850774797.1737335742,175,1.0,ETC,2025-06-11 09:50:48+09:00,2025-06-11 00:10:25.243602+00:00,2025-06-11 09:10:25.243602+09:00,0


In [ ]:
df_1stClick = df_1stClick.drop(columns=[
    'created_at',
    'is_migrated',
    'row_num_x',
    'row_num_y',
    'event_timestamp',
    'traffic_type',
    'user_pseudo_id_1',
    'user_id',
    'event_time_dt'
])


# 결과 전송

##대시보드 전송

In [ ]:
df_LastClick = df_LastClick.applymap(str)
df_LastClick['price'] = pd.to_numeric(df_LastClick['price'], errors='coerce').astype('Int64')


# --- 1. GCS 버킷에서 서비스 계정 정보 읽어오기 ---
# 클라우드 환경에 맞게 GCS 버킷에서 인증 정보를 가져옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# --- 2. gspread 인증 ---
# 읽어온 키 정보를 사용하여 인증합니다.
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# --- 3. 스프레드시트 및 워크시트 열기 ---
sheet_id = "1A6u7zZJQfOPmieQkzWtch-UsQH3xF3Hfj2vvyqyXjZg"
worksheet = gc.open_by_key(sheet_id).worksheet("U_LastClick")



# NaN 값을 빈 문자열로 바꿉니다.
df_for_upload = df_LastClick.fillna('')

# --- 5. 데이터 업데이트 ---
# 데이터프레임을 리스트 형태로 변환합니다 (헤더 포함).
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# A1 셀부터 데이터를 업데이트합니다.
worksheet.update('A1', data_to_upload)

print("스프레드시트 업데이트가 성공적으로 완료되었습니다.")


  df_LastClick = df_LastClick.applymap(str)

  worksheet.update('A1', data_to_upload)



스프레드시트 업데이트가 성공적으로 완료되었습니다.


In [ ]:
df_1stClick = df_1stClick.applymap(str)
df_1stClick['price'] = pd.to_numeric(df_1stClick['price'], errors='coerce').astype('Int64')


# --- 1. GCS 버킷에서 서비스 계정 정보 읽어오기 ---
# 클라우드 환경에 맞게 GCS 버킷에서 인증 정보를 가져옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# --- 2. gspread 인증 ---
# 읽어온 키 정보를 사용하여 인증합니다.
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# --- 3. 스프레드시트 및 워크시트 열기 ---
sheet_id = "1A6u7zZJQfOPmieQkzWtch-UsQH3xF3Hfj2vvyqyXjZg"
worksheet = gc.open_by_key(sheet_id).worksheet("U_1stClick")



# NaN 값을 빈 문자열로 바꿉니다.
df_for_upload = df_1stClick.fillna('')

# --- 5. 데이터 업데이트 ---
# 데이터프레임을 리스트 형태로 변환합니다 (헤더 포함).
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# A1 셀부터 데이터를 업데이트합니다.
worksheet.update('A1', data_to_upload)

print("스프레드시트 업데이트가 성공적으로 완료되었습니다.")


  df_1stClick = df_1stClick.applymap(str)

  worksheet.update('A1', data_to_upload)



스프레드시트 업데이트가 성공적으로 완료되었습니다.


## 운영노트 전송

In [ ]:

# --- 1. GCS 버킷에서 서비스 계정 정보 읽어오기 ---
# 클라우드 환경에 맞게 GCS 버킷에서 인증 정보를 가져옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# --- 2. gspread 인증 ---
# 읽어온 키 정보를 사용하여 인증합니다.
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# --- 3. 스프레드시트 및 워크시트 열기 ---
sheet_id = "1WUwD9wGaVm4wywiet9yFTbH7l-lbEWVZn5hu4rHOqFU"
worksheet = gc.open_by_key(sheet_id).worksheet("U_LastClick")



# NaN 값을 빈 문자열로 바꿉니다.
df_for_upload = df_LastClick.fillna('')

# --- 5. 데이터 업데이트 ---
# 데이터프레임을 리스트 형태로 변환합니다 (헤더 포함).
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# A1 셀부터 데이터를 업데이트합니다.
worksheet.update('A1', data_to_upload)

print("스프레드시트 업데이트가 성공적으로 완료되었습니다.")

  worksheet.update('A1', data_to_upload)



스프레드시트 업데이트가 성공적으로 완료되었습니다.


In [ ]:
df_1stClick = df_1stClick.applymap(str)
df_1stClick['price'] = pd.to_numeric(df_1stClick['price'], errors='coerce').astype('Int64')


# --- 1. GCS 버킷에서 서비스 계정 정보 읽어오기 ---
# 클라우드 환경에 맞게 GCS 버킷에서 인증 정보를 가져옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# --- 2. gspread 인증 ---
# 읽어온 키 정보를 사용하여 인증합니다.
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# --- 3. 스프레드시트 및 워크시트 열기 ---
sheet_id = "1WUwD9wGaVm4wywiet9yFTbH7l-lbEWVZn5hu4rHOqFU"
worksheet = gc.open_by_key(sheet_id).worksheet("U_1stClick")



# NaN 값을 빈 문자열로 바꿉니다.
df_for_upload = df_1stClick.fillna('')

# --- 5. 데이터 업데이트 ---
# 데이터프레임을 리스트 형태로 변환합니다 (헤더 포함).
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# A1 셀부터 데이터를 업데이트합니다.
worksheet.update('A1', data_to_upload)

print("스프레드시트 업데이트가 성공적으로 완료되었습니다.")


  df_1stClick = df_1stClick.applymap(str)

  worksheet.update('A1', data_to_upload)



스프레드시트 업데이트가 성공적으로 완료되었습니다.


In [ ]:
df_purchase.head()

,c_date,transaction_id,uid,status,product_name,start_date,initial_end_date,actual_end_date,price,created_at,...,event_timestamp,traffic_type,user_pseudo_id_1,user_id,row_num2,Media,created_at_dt,event_time_dt,event_time_kst,ConvDuration
3940,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,1749596034672101,None,161602292.1749596035,85,1.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:54.672101+00:00,2025-06-11 07:53:54.672101+09:00,0
3941,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,1749596034672101,None,161602292.1749596035,85,2.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:54.672101+00:00,2025-06-11 07:53:54.672101+09:00,0
3942,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,1749596034672101,None,161602292.1749596035,85,3.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:54.672101+00:00,2025-06-11 07:53:54.672101+09:00,0
3943,2025-06-11,166,85,WITHDRAW,무제한 패스 - 1개월,2025-06-11,2025-07-10,2025-06-11,0.0,2025-06-11 08:01:07,...,1749596036924766,None,161602292.1749596035,85,4.0,GoogleSEO,2025-06-11 08:01:07+09:00,2025-06-10 22:53:56.924766+00:00,2025-06-11 07:53:56.924766+09:00,0
7520,2025-06-11,255,122,CANCELED,2시간 패스 - 1회,None,None,None,5000.0,2025-06-11 09:02:23,...,1749597740144725,None,944027773.1743328104,122,1.0,ffapp,2025-06-11 09:02:23+09:00,2025-06-10 23:22:20.144725+00:00,2025-06-11 08:22:20.144725+09:00,0


# 구글시트에서 광고데이터 가져오기

In [ ]:
# 2. GCS에서 서비스 계정 정보를 읽어와 Google Sheets API를 인증합니다.
# GCS 버킷에서 인증 정보를 가져옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# GCS에서 읽어온 서비스 계정 정보로 gspread 인증
gc = gspread.service_account_from_dict(key_file_dict)


def get_spreadsheet_data(url, sheet_name):
    """지정된 URL과 시트 이름으로 Google Sheets 데이터를 가져와 DataFrame으로 반환합니다."""
    try:
        spreadsheet = gc.open_by_url(url)
        sheet = spreadsheet.worksheet(sheet_name)
        data = sheet.get_all_records()
        df = pd.DataFrame(data)
        return df
    except gspread.SpreadsheetNotFound:
        print(f"{url} 스프레드시트를 찾을 수 없습니다. URL과 시트 이름을 확인하세요.")
        raise
    except gspread.exceptions.APIError as e:
        print(f"API 오류 발생: {e}. 노트북의 서비스 계정이 스프레드시트에 접근할 권한이 있는지 확인하세요.")
        raise

def convert_to_integer(df, columns):
    """DataFrame의 특정 컬럼들을 정수형으로 변환합니다."""
    for column in columns:
        df[column] = pd.to_numeric(df[column], errors='coerce').fillna(0).astype(int)
    return df


# 3. 스프레드시트 데이터 가져오기 및 전처리
url2 = 'https://docs.google.com/spreadsheets/d/1iSzIoz6H31_NhaQr8prVTQWgKothQKgaRX0Y3v-qoN8/'
df_metaRaw = get_spreadsheet_data(url2, 'meta raw')
df_metaRaw = df_metaRaw[~((pd.isnull(df_metaRaw['Date'])) | (df_metaRaw['Date'].str.strip() == ''))]
df_metaRaw = df_metaRaw.replace(r"^\s*$", 0, regex=True)
df_metaRaw['Date'] = df_metaRaw['Date'].astype(str)
df_metaRaw['Ad url tags'] = df_metaRaw['Ad url tags'].astype(str)
df_metaRaw['Destination URL'] = df_metaRaw['Destination URL'].astype(str)
df_metaRaw['Cost'] = df_metaRaw['Cost'].astype(str)
df_metaRaw['Website purchases conversion value'] = df_metaRaw['Website purchases conversion value'].astype(str)
df_metaRaw['utm_content'] = df_metaRaw['Ad url tags'].str.extract(r'utm_content=([^&]+)')


df_metaRaw.tail()

,Date,Campaign name,Ad set name,Ad name,Ad url tags,Destination URL,Cost,Website purchases conversion value,utm_content
8172,2026-05-08,fbig-sns-purchase,FS_타겟메시지(1인사업자)-1_전환_구매_일반_mass_260504,fs2541-260424-ua-solo-founder-1-hwg,utm_source=fbig&utm_medium=display&utm_campaig...,https://fivespot.io/persona/solo-founder,1181,0,fs2541-260424-ua-solo-founder-1-hwg
8173,2026-05-08,fbig-sns-purchase,FS_타겟메시지(1인사업자)-1_전환_구매_일반_mass_260504,fs2543-260424-ua-solo-founder-3-hwg,utm_source=fbig&utm_medium=display&utm_campaig...,https://fivespot.io/persona/solo-founder,22880,0,fs2543-260424-ua-solo-founder-3-hwg
8174,2026-05-08,fbig-sns-purchase,FS_타겟메시지(1인사업자)-1_전환_구매_일반_mass_260504,fs2538-260424-ua-solo-founder-3-byn,utm_source=fbig&utm_medium=display&utm_campaig...,https://fivespot.io/persona/solo-founder,805,0,fs2538-260424-ua-solo-founder-3-byn
8175,2026-05-08,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs2546-260430-ua-weekly-1-ehs,utm_source=fbig&utm_medium=display&utm_campaig...,https://fivespot.io/event/weekly,857,0,fs2546-260430-ua-weekly-1-ehs
8176,2026-05-08,fbig-sns-purchase,FS_타겟메시지(프리랜서)-1_전환_구매_일반_mass_260406,fs2549-260430-ua-weekly-4-ehs,utm_source=fbig&utm_medium=display&utm_campaig...,https://fivespot.io/event/weekly,46086,0,fs2549-260430-ua-weekly-4-ehs


In [ ]:
url3 = 'https://docs.google.com/spreadsheets/d/1QBfuzJpzUwrJQzdsOm1lgNZjtWkQ3sz7FK07RnjkZlY/'
df_GoogleRaw = get_spreadsheet_data(url3, 'google raw')
df_GoogleRaw.head()

,Date,Campaign name,Ad group name,Ad ID,Cost,Total conversion value
0,2025-02-12,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732522123274,179225.21,0.0
1,2025-02-12,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732539357656,7733.73,0.0
2,2025-02-12,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732615085106,13789.52,0.0
3,2025-02-13,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732500739534,0.00,0.0
4,2025-02-13,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732522123274,96430.77,0.0


In [ ]:
url4 = 'https://docs.google.com/spreadsheets/d/19Nsrco3nMNngnTpXhGjQm4Dl01CxOpvUsgRcBd9Vm64/'
df_KakaoRaw = get_spreadsheet_data(url4, 'kakao raw')
df_KakaoRaw.head()

,날짜,캠페인명,광고그룹명,소재명,비용,회원가입,구매,소재ID
0,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2468-260213-ua-fffor1-1-ehs_500x500,20400,0,0,30103905
1,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2466-260213-ua-best-membership-3-ehs_800x1000,9042,0,0,30103923
2,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2463-260213-ua-main-1-ehs_1200x600,9010,0,0,30092879
3,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2466-260213-ua-best-membership-3-ehs_500x500,6943,0,0,30103925
4,2026-02-19,카카오_모먼트_전환_202602,kakao_visitor,fs2462-260213-re-weekly-1-ehs_1200x600,6034,0,0,30090916


In [ ]:
url5 = 'https://docs.google.com/spreadsheets/d/19Nsrco3nMNngnTpXhGjQm4Dl01CxOpvUsgRcBd9Vm64/'
df_gfaRaw = get_spreadsheet_data(url5, 'gfa raw')
df_gfaRaw.head()

,날짜,캠페인,광고그룹,광고소재,총비용,총전환수,총전환매출액,캠페인ID,광고그룹ID,광고소재ID
0,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2532-260423-ua-weekly-6-ehs,3051.818182,0,0,1307769,4192548,35245183
1,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2531-260423-ua-weekly-5-ehs,15367.272730,0,0,1307769,4192548,35245147
2,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2530-260423-ua-weekly-4-ehs,5925.454545,0,0,1307769,4192548,35245047
3,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2529-260423-ua-weekly-3-ehs,3356.363636,0,0,1307769,4192548,35244913
4,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2528-260423-ua-weekly-2-ehs,2970.000000,0,0,1307769,4192548,35244877


In [ ]:
# 4. BigQuery에 업로드하기 위해 데이터프레임을 준비합니다.
def rename_columns(df):
    """BigQuery 명명 규칙에 맞게 컬럼 이름을 변경합니다."""
    return df.rename(columns=lambda x: x.strip().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_"))

df_metaRaw = rename_columns(df_metaRaw)

# --- 데이터 타입 수정 부분 (이 코드를 추가하세요) ---
# 에러가 발생한 Campaign_name을 포함하여, 텍스트 데이터가 들어가는 컬럼들을 문자열 타입으로 강제 변환합니다.
cols_to_fix = ['Campaign_name', 'Ad_set_name', 'Ad_name', 'utm_content', 'Ad_url_tags', 'Destination_URL']

for col in cols_to_fix:
    if col in df_metaRaw.columns:
        # 1. 모든 데이터를 문자열로 변환
        df_metaRaw[col] = df_metaRaw[col].astype(str)
        # 2. 'nan' 문자열로 변환된 결측치를 실제 None(Null)으로 변경 (선택 사항이지만 권장)
        df_metaRaw[col] = df_metaRaw[col].replace('nan', None)

print("\n--- 데이터 타입 변환 및 컬럼명 변경 완료 ---")
print(df_metaRaw.info()) # 데이터 타입이 object로 잘 나오는지 확인용
# ----------------------------------------------

# 5. BigQuery에 데이터를 업로드합니다.
bq_client = bigquery.Client.from_service_account_info(
    key_file_dict,
    project='fivespot-bigquery'
)

dataset_id = 'MKT'
table_id = 'meta_raw'
table_ref = bq_client.dataset(dataset_id).table(table_id)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = bq_client.load_table_from_dataframe(
    df_metaRaw, table_ref, job_config=job_config
)

job.result()
print(f"\n{job.output_rows}개의 행을 {dataset_id}.{table_id} 테이블에 성공적으로 업로드했습니다.")


--- 데이터 타입 변환 및 컬럼명 변경 완료 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8177 entries, 0 to 8176
Data columns (total 9 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Date                                8177 non-null   object
 1   Campaign_name                       8177 non-null   object
 2   Ad_set_name                         8177 non-null   object
 3   Ad_name                             8177 non-null   object
 4   Ad_url_tags                         8177 non-null   object
 5   Destination_URL                     8177 non-null   object
 6   Cost                                8177 non-null   object
 7   Website_purchases_conversion_value  8177 non-null   object
 8   utm_content                         8099 non-null   object
dtypes: object(9)
memory usage: 575.1+ KB
None

8177개의 행을 MKT.meta_raw 테이블에 성공적으로 업로드했습니다.


In [ ]:
# 6. BigQuery에서 통합 데이터를 쿼리하고 처리합니다.
print("\n--- BigQuery에서 통합 데이터 쿼리 시작 ---")
sql_query = """
SELECT * FROM `fivespot-bigquery.MKT.meta_fix`
UNION ALL
SELECT * FROM `fivespot-bigquery.MKT.meta_raw`
ORDER BY Date
"""

# 쿼리 실행 및 데이터프레임으로 변환
df_meta = bq_client.query(sql_query).to_dataframe()
df_meta = df_meta[df_meta['utm_content'].notna() & (df_meta['utm_content'] != '')] # utm태그 누락 행 제거

# 데이터 타입 변환
df_meta['Cost'] = pd.to_numeric(df_meta['Cost'], errors='coerce')
df_meta['Website_purchases_conversion_value'] = pd.to_numeric(df_meta['Website_purchases_conversion_value'], errors='coerce')

# 인덱스 생성 (필요 시 사용)
meta_index = df_meta[['Campaign_name', 'Ad_set_name', 'utm_content']].drop_duplicates().reset_index(drop=True)

# Groupby 및 집계 수행
df_meta_grouped = (
    df_meta.groupby(['Date', 'Campaign_name', 'Ad_set_name', 'utm_content'], as_index=False)
    .agg({
        'Cost': 'sum',
        'Website_purchases_conversion_value': 'sum'
    })
)

# 컬럼명 변경
df_meta_grouped.rename(columns={'Website_purchases_conversion_value': '매체거래액'}, inplace=True)

# Date 컬럼을 날짜 형식으로 변환 후 필터링
df_meta_grouped['Date'] = pd.to_datetime(df_meta_grouped['Date'], errors='coerce')
df_meta_grouped = df_meta_grouped[df_meta_grouped['Date'] >= pd.to_datetime('2025-06-16')]

print("\n--- 최종 데이터 처리 완료 ---")
df_meta_grouped.head()


--- BigQuery에서 통합 데이터 쿼리 시작 ---

--- 최종 데이터 처리 완료 ---


,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액
16521,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1932-250313-re-fffor1-1-ehs,138304,263150
16522,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1984-250410-ua-fffor1-6-lyj,8805,0
16523,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1988-250410-ua-fffor1-10-lyj,48353,0
16524,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1972-250402-ua-foolsday-7-iyl,78172,0
16525,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1973-250402-ua-foolsday-8-iyl,121867,0


In [ ]:
df_GoogleRaw.rename(columns={'Campaign name': 'Campaign_name'}, inplace=True)
df_GoogleRaw.rename(columns={'Ad group name': 'Ad_set_name'}, inplace=True)
df_GoogleRaw.rename(columns={'Ad ID': 'utm_content'}, inplace=True)
df_GoogleRaw.rename(columns={'Total conversion value': '매체거래액'}, inplace=True)

df_GoogleRaw.head()

,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액
0,2025-02-12,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732522123274,179225.21,0.0
1,2025-02-12,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732539357656,7733.73,0.0
2,2025-02-12,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732615085106,13789.52,0.0
3,2025-02-13,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732500739534,0.00,0.0
4,2025-02-13,FS_첫구매자프로모션_디스커버리_전환_구매_프로모션_250212_GDN,시그널조합,732522123274,96430.77,0.0


In [ ]:
df_GoogleRaw['Date'] = pd.to_datetime(df_GoogleRaw['Date'], errors='coerce')
df_GoogleRaw['utm_content'] = df_GoogleRaw['utm_content'].astype(str)
df_GoogleRaw['Cost'] = df_GoogleRaw['Cost'].astype(int)
df_GoogleRaw = df_GoogleRaw[df_GoogleRaw['Date'] >= pd.to_datetime('2025-06-16')]


In [ ]:
df_KakaoRaw.head()

,날짜,캠페인명,광고그룹명,소재명,비용,회원가입,구매,소재ID
0,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2468-260213-ua-fffor1-1-ehs_500x500,20400,0,0,30103905
1,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2466-260213-ua-best-membership-3-ehs_800x1000,9042,0,0,30103923
2,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2463-260213-ua-main-1-ehs_1200x600,9010,0,0,30092879
3,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2466-260213-ua-best-membership-3-ehs_500x500,6943,0,0,30103925
4,2026-02-19,카카오_모먼트_전환_202602,kakao_visitor,fs2462-260213-re-weekly-1-ehs_1200x600,6034,0,0,30090916


In [ ]:
df_KakaoRaw.rename(columns={'날짜': 'Date'}, inplace=True)
df_KakaoRaw.rename(columns={'캠페인명': 'Campaign_name'}, inplace=True)
df_KakaoRaw.rename(columns={'광고그룹명': 'Ad_set_name'}, inplace=True)
df_KakaoRaw.rename(columns={'소재명': 'utm_content'}, inplace=True)
df_KakaoRaw.rename(columns={'비용': 'Cost'}, inplace=True)
df_KakaoRaw['매체거래액'] = 0
df_KakaoRaw.head()

,Date,Campaign_name,Ad_set_name,utm_content,Cost,회원가입,구매,소재ID,매체거래액
0,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2468-260213-ua-fffor1-1-ehs_500x500,20400,0,0,30103905,0
1,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2466-260213-ua-best-membership-3-ehs_800x1000,9042,0,0,30103923,0
2,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2463-260213-ua-main-1-ehs_1200x600,9010,0,0,30092879,0
3,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2466-260213-ua-best-membership-3-ehs_500x500,6943,0,0,30103925,0
4,2026-02-19,카카오_모먼트_전환_202602,kakao_visitor,fs2462-260213-re-weekly-1-ehs_1200x600,6034,0,0,30090916,0


In [ ]:
import pandas as pd

# 1. utm_content에서 '_숫자x숫자' 패턴 제거 (정규표현식 사용)
# _ 뒤에 숫자들(d+)이 오고 x가 온 뒤 다시 숫자들이 오는 패턴을 찾아 삭제합니다.
df_KakaoRaw['utm_content'] = df_KakaoRaw['utm_content'].str.replace(r'_\d+x\d+', '', regex=True)

# 2. 필요한 컬럼만 선택하고 그룹바이 수행
# 기준 컬럼: Date, Campaign_name, Ad_set_name, utm_content
# 합산 컬럼: Cost, 매체거래액
df_KakaoRaw_2 = df_KakaoRaw.groupby(
    ['Date', 'Campaign_name', 'Ad_set_name', 'utm_content'],
    as_index=False
)[['Cost', '매체거래액']].sum()

# 결과 확인
df_KakaoRaw_2.head()

,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액
0,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2463-260213-ua-main-1-ehs,14936,0
1,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2466-260213-ua-best-membership-3-ehs,20259,0
2,2026-02-19,카카오_모먼트_전환_202602,kakao_mass,fs2468-260213-ua-fffor1-1-ehs,30342,0
3,2026-02-19,카카오_모먼트_전환_202602,kakao_visitor,fs2462-260213-re-weekly-1-ehs,15012,0
4,2026-02-19,카카오_모먼트_전환_202602,kakao_visitor,fs2467-260213-re-weekly-2-ehs,10488,0


In [ ]:
# Date 컬럼을 datetime 타입으로 변경
df_KakaoRaw_2['Date'] = pd.to_datetime(df_KakaoRaw_2['Date'])

# 매체거래액 컬럼을 float 타입으로 변경
df_KakaoRaw_2['매체거래액'] = df_KakaoRaw_2['매체거래액'].astype(float)

In [ ]:
df_gfaRaw.head()

,날짜,캠페인,광고그룹,광고소재,총비용,총전환수,총전환매출액,캠페인ID,광고그룹ID,광고소재ID
0,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2532-260423-ua-weekly-6-ehs,3051.818182,0,0,1307769,4192548,35245183
1,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2531-260423-ua-weekly-5-ehs,15367.272730,0,0,1307769,4192548,35245147
2,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2530-260423-ua-weekly-4-ehs,5925.454545,0,0,1307769,4192548,35245047
3,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2529-260423-ua-weekly-3-ehs,3356.363636,0,0,1307769,4192548,35244913
4,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2528-260423-ua-weekly-2-ehs,2970.000000,0,0,1307769,4192548,35244877


In [ ]:
df_gfaRaw.rename(columns={'날짜': 'Date'}, inplace=True)
df_gfaRaw.rename(columns={'캠페인': 'Campaign_name'}, inplace=True)
df_gfaRaw.rename(columns={'광고그룹': 'Ad_set_name'}, inplace=True)
df_gfaRaw.rename(columns={'광고소재': 'utm_content'}, inplace=True)
df_gfaRaw.rename(columns={'총비용': 'Cost'}, inplace=True)
df_gfaRaw.rename(columns={'총전환매출액': '매체거래액'}, inplace=True)
df_gfaRaw = df_gfaRaw.drop(columns=['총전환수'])
df_gfaRaw.head()

,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액,캠페인ID,광고그룹ID,광고소재ID
0,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2532-260423-ua-weekly-6-ehs,3051.818182,0,1307769,4192548,35245183
1,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2531-260423-ua-weekly-5-ehs,15367.272730,0,1307769,4192548,35245147
2,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2530-260423-ua-weekly-4-ehs,5925.454545,0,1307769,4192548,35245047
3,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2529-260423-ua-weekly-3-ehs,3356.363636,0,1307769,4192548,35244913
4,2026-04-23,FS_전환_구매_일반_mass_260422,FS_키워드타겟팅_전환_구매_프로모션_mass_260422,fs2528-260423-ua-weekly-2-ehs,2970.000000,0,1307769,4192548,35244877


In [ ]:
# Date 컬럼을 datetime 타입으로 변경
df_gfaRaw['Date'] = pd.to_datetime(df_gfaRaw['Date'])

# 매체거래액 컬럼을 float 타입으로 변경
df_gfaRaw['매체거래액'] = df_gfaRaw['매체거래액'].astype(float)

In [ ]:
df_gfaRaw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           75 non-null     datetime64[ns]
 1   Campaign_name  75 non-null     object        
 2   Ad_set_name    75 non-null     object        
 3   utm_content    75 non-null     object        
 4   Cost           75 non-null     float64       
 5   매체거래액          75 non-null     float64       
 6   캠페인ID          75 non-null     int64         
 7   광고그룹ID         75 non-null     int64         
 8   광고소재ID         75 non-null     int64         
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 5.4+ KB


In [ ]:
df_gfaRaw = df_gfaRaw.groupby(
    ['Date', 'Campaign_name', 'Ad_set_name', 'utm_content'],
    as_index=False
)[['Cost', '매체거래액']].sum()

In [ ]:
df_DA_MediaRaw = pd.concat([df_meta_grouped, df_GoogleRaw, df_KakaoRaw_2, df_gfaRaw], ignore_index=True).drop_duplicates()
df_DA_MediaRaw.head()

,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액
0,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1932-250313-re-fffor1-1-ehs,138304.0,263150.0
1,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1984-250410-ua-fffor1-6-lyj,8805.0,0.0
2,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1988-250410-ua-fffor1-10-lyj,48353.0,0.0
3,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1972-250402-ua-foolsday-7-iyl,78172.0,0.0
4,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1973-250402-ua-foolsday-8-iyl,121867.0,0.0


# DA only dataDriven

In [ ]:
import pandas as pd

# 1. 기본 필터링 (매체 선택 및 첫구매 기준)
df_DA = df_purchase[df_purchase['Media'].isin(['FBIG', 'GoogleDA', 'Kakao', 'NaverGFA'])].copy()
df_DA = df_DA[df_DA['purchase_type'] == '첫구매']

# 2. row_num2 숫자 변환 (순서 파악 용도)
df_DA['row_num2'] = pd.to_numeric(df_DA['row_num2'], errors='coerce')

# ---------------------------------------------------------
# [모델 1] Last Click (마지막 클릭)
# ---------------------------------------------------------
max_row_num2 = df_DA.groupby('uid')['row_num2'].transform('max')
df_LastClick = df_DA[df_DA['row_num2'] == max_row_num2].copy()

df_LastClick_grouped = df_LastClick.groupby(['c_date', 'event_content'])[['uid', 'price']].agg({'uid': 'nunique', 'price': 'sum'})
df_LastClick_grouped = df_LastClick_grouped.rename(columns={'uid': '구매수LastClick', 'price': '거래액LastClick'}).reset_index()

# ---------------------------------------------------------
# [모델 2] 1st Click (첫 클릭 / 기존 코드의 7days)
# ---------------------------------------------------------
min_row_num2 = df_DA.groupby('uid')['row_num2'].transform('min')
df_1stClick = df_DA[df_DA['row_num2'] == min_row_num2].copy()

df_1stClick_grouped = df_1stClick.groupby(['c_date', 'event_content'])[['uid', 'price']].agg({'uid': 'nunique', 'price': 'sum'})
df_1stClick_grouped = df_1stClick_grouped.rename(columns={'uid': '구매수7days', 'price': '거래액7days'}).reset_index()

# ---------------------------------------------------------
# [모델 3] Linear Click (선형 기여 - 신규 추가)
# ---------------------------------------------------------
# 조건: 동일소재 두번 본 것은 1번만 카운팅
df_linear_base = df_DA.drop_duplicates(subset=['uid', 'event_content']).copy()

# 각 uid별로 기여도 분모(n) 구하기 (한 유저가 본 고유 소재의 개수)
df_linear_base['n'] = df_linear_base.groupby('uid')['event_content'].transform('count')

# 기여도 계산 (1건을 n으로 나누고, 거래액을 n으로 나눔)
df_linear_base['구매수Linear'] = 1 / df_linear_base['n']
df_linear_base['거래액Linear'] = df_linear_base['price'] / df_linear_base['n']

# 소재별 합계 집계
df_Linear_grouped = df_linear_base.groupby(['c_date', 'event_content'])[['구매수Linear', '거래액Linear']].sum().reset_index()

# ---------------------------------------------------------
# 데이터 병합 (LastClick + 1stClick + Linear)
# ---------------------------------------------------------
# 1. 모델간 병합
merged_df_DA = pd.merge(
    df_LastClick_grouped,
    df_1stClick_grouped,
    how='outer',
    on=['c_date', 'event_content']
)

merged_df_DA = pd.merge(
    merged_df_DA,
    df_Linear_grouped,
    how='outer',
    on=['c_date', 'event_content']
)

# 2. 메타 정보(Campaign, Ad_set 등) 인덱스 생성 및 병합
df_DA_MediaRaw_index = df_DA_MediaRaw[['utm_content', 'Campaign_name', 'Ad_set_name']].drop_duplicates(subset='utm_content', keep='first')

merged_df_DA = merged_df_DA.merge(
    df_DA_MediaRaw_index,
    how='left',
    left_on='event_content',
    right_on='utm_content'
)

# 3. 데이터 정제
merged_df_DA.fillna(0, inplace=True)
merged_df_DA = merged_df_DA[merged_df_DA['utm_content'] != 0]
merged_df_DA['c_date'] = pd.to_datetime(merged_df_DA['c_date'], errors='coerce')

# 4. 원본 MediaRaw 데이터와 최종 병합
merged_df = df_DA_MediaRaw.merge(
    merged_df_DA,
    how='outer',
    left_on=['Date', 'Campaign_name', 'Ad_set_name', 'utm_content'],
    right_on=['c_date', 'Campaign_name', 'Ad_set_name', 'utm_content']
)

# 5. 후처리 (날짜 채우기 및 불필요 컬럼 제거)
merged_df['Date'].fillna(merged_df['c_date'], inplace=True)
merged_df.fillna(0, inplace=True)
merged_df.drop(['c_date', 'event_content'], axis=1, errors='ignore', inplace=True)

# 최종 결과 확인
merged_df.head()

The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Date'].fillna(merged_df['c_date'], inplace=True)

  merged_df.fillna(0, inplace=True)



,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액,구매수LastClick,거래액LastClick,구매수7days,거래액7days,구매수Linear,거래액Linear
0,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1932-250313-re-fffor1-1-ehs,138304.0,263150.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1984-250410-ua-fffor1-6-lyj,8805.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1988-250410-ua-fffor1-10-lyj,48353.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1972-250402-ua-foolsday-7-iyl,78172.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1973-250402-ua-foolsday-8-iyl,121867.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 숫자형 컬럼들 중에서 float이지만 정수로 표현 가능한 경우 int로 변환
for col in merged_df.select_dtypes(include='number').columns:
    # NaN 값 있는 경우 int 변환이 불가능하므로 우선 채움 또는 스킵 필요
    if merged_df[col].isnull().any():
        continue  # 또는: merged_df[col] = merged_df[col].fillna(0)

    # 모든 값이 정수처럼 보이면 int로 변환
    if (merged_df[col] % 1 == 0).all():
        merged_df[col] = merged_df[col].astype(int)
merged_df.head()

,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액,구매수LastClick,거래액LastClick,구매수7days,거래액7days,구매수Linear,거래액Linear
0,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1932-250313-re-fffor1-1-ehs,138304.0,263150.0,0,0,0,0,0.0,0.0
1,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1984-250410-ua-fffor1-6-lyj,8805.0,0.0,0,0,0,0,0.0,0.0
2,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1988-250410-ua-fffor1-10-lyj,48353.0,0.0,0,0,0,0,0.0,0.0
3,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1972-250402-ua-foolsday-7-iyl,78172.0,0.0,0,0,0,0,0.0,0.0
4,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1973-250402-ua-foolsday-8-iyl,121867.0,0.0,0,0,0,0,0.0,0.0


## 대시보드 전송

In [ ]:
# # 1. 필요한 라이브러리를 임포트합니다.
# import pandas as pd
# import gspread
# from google.cloud import storage
# import json
# from datetime import datetime

# 2. GCS 버킷에서 서비스 계정 정보를 읽어옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# 3. gspread를 인증합니다.
gc = gspread.service_account_from_dict(key_file_dict)

# 4. 스프레드시트 및 워크시트를 엽니다.
sheet_id = "1A6u7zZJQfOPmieQkzWtch-UsQH3xF3Hfj2vvyqyXjZg"
worksheet = gc.open_by_key(sheet_id).worksheet("U_tableau raw")

# --- 이 코드는 이전 단계에서 `merged_df`라는 데이터프레임이
# --- 이미 생성되었다고 가정하고 실행됩니다.

# 5. 업로드를 위해 데이터프레임을 준비합니다.
print("\n--- Google Sheets 업로드 시작 ---")

# 업로드를 위해 데이터프레임 복사
df_for_upload = merged_df.copy()

# TypeError 방지를 위해 날짜/시간 컬럼을 문자열로 변환합니다.
if 'Date' in df_for_upload.columns and pd.api.types.is_datetime64_any_dtype(df_for_upload['Date']):
    df_for_upload['Date'] = df_for_upload['Date'].dt.strftime('%Y-%m-%d')

# NaN 값을 업로드 가능한 빈 문자열로 변경합니다.
df_for_upload = df_for_upload.fillna('')

# 6. 데이터를 Google Sheets에 업데이트합니다.
# 데이터프레임을 리스트 형태로 변환합니다 (헤더 포함).
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# A1 셀부터 데이터를 업데이트합니다.
worksheet.update(range_name='A1', values=data_to_upload)

print("스프레드시트 업데이트가 성공적으로 완료되었습니다.")



--- Google Sheets 업로드 시작 ---
스프레드시트 업데이트가 성공적으로 완료되었습니다.


## 운영노트 전송

In [ ]:


# 2. GCS 버킷에서 서비스 계정 정보를 읽어옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# 3. gspread를 인증합니다.
gc = gspread.service_account_from_dict(key_file_dict)

# 4. 스프레드시트 및 워크시트를 엽니다.
sheet_id = "1WUwD9wGaVm4wywiet9yFTbH7l-lbEWVZn5hu4rHOqFU"
worksheet = gc.open_by_key(sheet_id).worksheet("U_tableau raw")

# --- 이 코드는 이전 단계에서 `merged_df`라는 데이터프레임이
# --- 이미 생성되었다고 가정하고 실행됩니다.

# 5. 업로드를 위해 데이터프레임을 준비합니다.
print("\n--- Google Sheets 업로드 시작 ---")

# 업로드를 위해 데이터프레임 복사
df_for_upload = merged_df.copy()

# TypeError 방지를 위해 날짜/시간 컬럼을 문자열로 변환합니다.
if 'Date' in df_for_upload.columns and pd.api.types.is_datetime64_any_dtype(df_for_upload['Date']):
    df_for_upload['Date'] = df_for_upload['Date'].dt.strftime('%Y-%m-%d')

# NaN 값을 업로드 가능한 빈 문자열로 변경합니다.
df_for_upload = df_for_upload.fillna('')

# 6. 데이터를 Google Sheets에 업데이트합니다.
# 데이터프레임을 리스트 형태로 변환합니다 (헤더 포함).
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# A1 셀부터 데이터를 업데이트합니다.
worksheet.update(range_name='A1', values=data_to_upload)

print("스프레드시트 업데이트가 성공적으로 완료되었습니다.")



--- Google Sheets 업로드 시작 ---
스프레드시트 업데이트가 성공적으로 완료되었습니다.


##빅쿼리 최종데이터 업로드

In [ ]:

# 2. GCS 버킷에서 서비스 계정 정보를 읽어옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# --- 이 코드는 이전 단계에서 `merged_df`라는 데이터프레임이
# --- 이미 생성되었다고 가정하고 실행됩니다.

# 3. BigQuery에 업로드하기 위해 데이터프레임을 준비합니다.
# 모든 셀의 데이터를 문자열로 변환합니다.
merged_df = merged_df.applymap(str)
# Date 컬럼의 형식을 'YYYY-MM-DD'로 맞춥니다.
merged_df['Date'] = pd.to_datetime(merged_df['Date']).dt.strftime('%Y-%m-%d')

# BigQuery 명명 규칙에 맞게 컬럼 이름을 변경하는 함수를 정의합니다.
def rename_columns(df):
    return df.rename(columns=lambda x: x.strip().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_"))

merged_df = rename_columns(merged_df)

# 4. BigQuery 클라이언트를 초기화하고 데이터를 업로드합니다.
print("\n--- BigQuery 업로드 시작 ---")

# GCS에서 가져온 서비스 계정 정보로 BigQuery 클라이언트를 초기화합니다.
bq_client = bigquery.Client.from_service_account_info(
    key_file_dict,
    project='fivespot-bigquery'
)

# 데이터셋 및 테이블 정보를 정의합니다.
dataset_id = 'MKT'
table_id = 'U_DA_raw'
table_ref = bq_client.dataset(dataset_id).table(table_id)

# 테이블에 데이터를 덮어쓰도록 로드 작업을 설정합니다.
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# 데이터를 50,000행씩 나누어 BigQuery 테이블로 업로드합니다.
chunk_size = 50000
for i in range(0, len(merged_df), chunk_size):
    chunk = merged_df.iloc[i:i + chunk_size]
    job = bq_client.load_table_from_dataframe(chunk, table_ref, job_config=job_config)
    job.result()  # 작업이 완료될 때까지 기다립니다.
    print(f"Uploaded chunk {i // chunk_size + 1}.")

print(f"\n모든 데이터를 {dataset_id}.{table_id} 테이블에 성공적으로 업로드했습니다.")


  merged_df = merged_df.applymap(str)




--- BigQuery 업로드 시작 ---
Uploaded chunk 1.

모든 데이터를 MKT.U_DA_raw 테이블에 성공적으로 업로드했습니다.


In [ ]:
merged_df.head()

,Date,Campaign_name,Ad_set_name,utm_content,Cost,매체거래액,구매수LastClick,거래액LastClick,구매수7days,거래액7days,구매수Linear,거래액Linear
0,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1932-250313-re-fffor1-1-ehs,138304.0,263150.0,0,0,0,0,0.0,0.0
1,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1984-250410-ua-fffor1-6-lyj,8805.0,0.0,0,0,0,0,0.0,0.0
2,2025-06-16,fbig-sns-purchase,FS_1인리드프로모션-1_전환_구매_프로모션_패파SF1-2_250120,fs1988-250410-ua-fffor1-10-lyj,48353.0,0.0,0,0,0,0,0.0,0.0
3,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1972-250402-ua-foolsday-7-iyl,78172.0,0.0,0,0,0,0,0.0,0.0
4,2025-06-16,fbig-sns-purchase,FS_3천원프로모션-vari-1_전환_구매_프로모션_mass_250328,fs1973-250402-ua-foolsday-8-iyl,121867.0,0.0,0,0,0,0,0.0,0.0
